# Proyecto Final: Análisis del comportamiento de una aplicación bajo diferentes escenarios de despliegue

**Curso:** Principios y prácticas de Desarrollo de Software Orientado a Objetos  
**Instructor:** Gabriel Rodrigo Pedraza Ferreira  
**Integrantes:** `[COMPLETAR: nombres]`  
**Repositorio:** `[COMPLETAR: URL de GitHub/GitLab]`  
**Fecha:** `[COMPLETAR]`

---

### Contenido
1. Introducción y objetivos
2. Descripción de la aplicación
3. Entorno experimental
4. Configuración de despliegue por escenario (Docker Compose, Kubernetes 1/2/3 nodos)
5. Monitorización (opcional: Prometheus y Grafana)
6. Metodología de carga con JMeter
7. Carga de los resultados
8. Resultados: tiempo medio de respuesta y throughput
9. Análisis comparativo
10. Conclusiones
11. Recomendaciones
12. Limitaciones y amenazas a la validez

## 1. Introducción y objetivos

Este proyecto estudia de forma **empírica** cómo cambia el comportamiento de una misma aplicación
cuando se despliega en entornos distintos: una sola máquina con Docker Compose y clústeres de
Kubernetes de uno, dos y tres nodos, variando además el número de **réplicas** de la API.
Se mide con **Apache JMeter** el *tiempo medio de respuesta* y el *throughput* bajo distintos
niveles de carga concurrente, y se analizan los datos con Python en este notebook.

**Objetivo general.** Analizar el comportamiento de la aplicación bajo diferentes escenarios de despliegue,
observando métricas de rendimiento y escalabilidad para comprender las implicaciones de cada configuración,
en especial del escalado horizontal.

**Objetivos específicos**
- Desplegar la aplicación con Docker Compose en una sola máquina.
- Desplegarla en Kubernetes con 1, 2 y 3 nodos, variando las réplicas (1, 2 y 3).
- Generar carga con JMeter y registrar tiempo medio de respuesta y throughput en cada escenario.
- Comparar escenarios y configuraciones de réplicas, e identificar tendencias y cuellos de botella.
- Extraer conclusiones sobre escalabilidad, eficiencia de recursos y limitaciones.
- *(Opcional)* Observar CPU y memoria con Prometheus/Grafana.

**Hipótesis de partida** (se contrastan con los datos en las secciones 9 y 10):
1. El throughput crece al añadir réplicas mientras el nodo disponga de CPU libre para ellas.
2. En un solo nodo, añadir réplicas por encima de los núcleos disponibles aporta poco o nada.
3. Con más nodos el escalado horizontal rinde más, hasta que otro componente (la base de datos) se convierte en el cuello de botella.
4. Cuando el sistema se satura, el tiempo de respuesta crece de forma aproximadamente lineal con el número de usuarios.

## 2. Descripción de la aplicación

**Tienda API** es un servicio *backend* con **API REST** (FastAPI, Python 3.12) y persistencia en **PostgreSQL 16**.
Modela un pequeño comercio electrónico: catálogo de productos, clientes y pedidos.

### Arquitectura por capas (orientada a objetos)

```
HTTP (routers FastAPI)  ->  Servicios (reglas de negocio)  ->  Puertos (interfaces ABC)  ->  Repositorios PostgreSQL
 app/routers/*.py           app/services.py                    app/ports.py                  app/repositories_pg.py
                                         \__ Entidades de dominio inmutables: app/domain.py
 Raíz de composición: app/container.py        Pool de conexiones: app/database.py
```

- **Responsabilidad única:** routers (HTTP), servicios (reglas), repositorios (SQL), `Database` (conexiones).
- **Inversión de dependencias:** los servicios dependen de interfaces (`ports.py`), no de PostgreSQL, por lo que se prueban con repositorios en memoria (`tests/`).
- **Inyección de dependencias:** `Container` construye y conecta todas las piezas en un único lugar.

### Endpoints

| Método | Ruta | Descripción | Tipo de carga |
|---|---|---|---|
| GET | `/products?page=&size=&category_id=` | Listado paginado | lectura + `COUNT(*)` |
| GET | `/products/{id}` | Detalle de producto | lectura por clave primaria |
| GET | `/products/search?q=` | Búsqueda `ILIKE` sin índice | lectura con escaneo secuencial |
| POST | `/orders` | Crea pedido y líneas en una transacción | **escritura** |
| GET | `/orders/{id}`, `/customers/{id}/orders` | Consulta de pedidos | lectura con joins |
| GET | `/reports/top-products`, `/reports/sales-by-category` | Agregaciones sobre `order_items` | lectura **pesada** |
| GET | `/reports/counts` | Conteo de registros por tabla | verificación del dataset |
| GET | `/health`, `/ready`, `/info`, `/metrics` | Operación, trazabilidad de réplica y métricas Prometheus | — |

### Datos (requisito: mínimo 10.000 registros)

| Tabla | Registros |
|---|---:|
| `categories` | 20 |
| `products` | 20.000 |
| `customers` | 10.000 |
| `orders` | 30.000 |
| `order_items` | ≈ 90.000 |
| **Total** | **≈ 150.000** |

Los datos se generan de forma determinista con `db/seed.sql` (`generate_series`), y se cargan con `python -m app.seed`
(servicio `seed` en Compose y `Job` en Kubernetes). El endpoint `/reports/counts` permite comprobar la cantidad real de registros.

In [ ]:
# ---- Configuración general (edita solo esta celda) -------------------------------------------
REPO_URL = "https://github.com/TU_USUARIO/proyecto-final-despliegue.git"   # <- cambia por tu repositorio
USE_DEMO_DATA = False        # True = genera datos SINTÉTICOS solo para probar el notebook
RESULTS_DIR = "results"      # carpeta con los .jtl reales medidos con JMeter
# ------------------------------------------------------------------------------------------------

import os, re, subprocess, sys, json, glob, urllib.parse, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.width", 160)


def locate_repo():
    "Se ubica en la raíz del repositorio (local, o clonándolo si se ejecuta en Colab)."
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "k8s").is_dir() and (candidate / "app").is_dir():
            return candidate
    if "TU_USUARIO" in REPO_URL:
        raise RuntimeError("Edita REPO_URL en la primera celda para poder clonar tu repositorio.")
    target = Path("proyecto-final-despliegue")
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    return target.resolve()


REPO = locate_repo()
os.chdir(REPO)
print("Repositorio:", REPO)


def show_file(path, lang=""):
    "Muestra un archivo del repositorio (configuración de despliegue) dentro del notebook."
    text = Path(path).read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
        display(Markdown(f"**`{path}`**\n\n```{lang}\n{text}\n```"))
    except Exception:
        print(f"--- {path} ---\n{text}")


if USE_DEMO_DATA:
    subprocess.run([sys.executable, "scripts/generate_demo_results.py", "--out", "results_demo"], check=True)
    RESULTS_DIR = "results_demo"
    WARMUP_S = 10      # los datos demo usan 10 s de ramp-up
    TAG = "  [DATOS SINTÉTICOS DE DEMOSTRACIÓN]"
    print("ATENCIÓN: se están usando datos sintéticos. No los uses en el informe final.")
else:
    WARMUP_S = 30      # = RAMP de scripts/run_jmeter.sh (se descartan los primeros 30 s de cada prueba)
    TAG = ""

## 3. Entorno experimental

> Completa esta tabla con las características **reales** de tus máquinas. Es parte de la rigurosidad del experimento:
> sin conocer el hardware no se pueden interpretar los resultados.

| Rol | Máquina / proveedor | vCPU | RAM | SO | Notas |
|---|---|---:|---:|---|---|
| Servidor Docker Compose | `[COMPLETAR]` | `[ ]` | `[ ]` | `[ ]` | Debe ser distinta de la máquina de JMeter |
| Nodo 1 de Kubernetes (control-plane + PostgreSQL) | `[COMPLETAR]` | `[ ]` | `[ ]` | `[ ]` | Etiquetado `role=database` |
| Nodo 2 de Kubernetes | `[COMPLETAR]` | `[ ]` | `[ ]` | `[ ]` | |
| Nodo 3 de Kubernetes | `[COMPLETAR]` | `[ ]` | `[ ]` | `[ ]` | |
| Generador de carga (JMeter) | `[COMPLETAR]` | `[ ]` | `[ ]` | `[ ]` | Misma red que los servidores |

| Software | Versión |
|---|---|
| Docker / Docker Compose | `[COMPLETAR]` |
| K3s (o MicroK8s / Kind) | `[COMPLETAR]` |
| Apache JMeter | `[COMPLETAR]` |
| Python / FastAPI / PostgreSQL | 3.12 / 0.115 / 16 |

**Límites de recursos por réplica (idénticos en Compose y Kubernetes):** 0,5 vCPU y 512 MiB para la API; PostgreSQL con hasta 2 vCPU y 2 GiB.

## 4. Configuración de despliegue por escenario

La **misma imagen de contenedor** (`Dockerfile`) se usa en todos los escenarios. Solo cambia el entorno que la ejecuta.

### 4.1 Fase 1: Docker Compose (una sola máquina)
`docker-compose.yml` define tres servicios: `db` (PostgreSQL con *healthcheck*), `seed` (tarea de una ejecución que crea
el esquema y carga los datos) y `api` (con límites de 0,5 CPU / 512 MiB, el mismo recurso que un pod de Kubernetes).

In [ ]:
show_file("docker-compose.yml", "yaml")
show_file("Dockerfile", "dockerfile")

### 4.2 Fases 2–4: Kubernetes con 1, 2 y 3 nodos

Manifiestos en `k8s/` (se aplican con `kubectl apply -k k8s/`):

| Archivo | Contenido |
|---|---|
| `00-namespace.yaml` | Namespace `tienda` |
| `01-config.yaml` | `ConfigMap` (conexión a BD, tamaño del pool) y `Secret` (contraseñas) |
| `02-postgres.yaml` | PostgreSQL: `PersistentVolumeClaim`, `Deployment` (1 réplica) y `Service` interno |
| `03-seed-job.yaml` | `Job` que crea el esquema y carga los >150.000 registros |
| `04-api.yaml` | `Deployment` de la API (réplicas variables) y `Service` NodePort `30080` |
| `kustomization.yaml` | Une todo y fija la imagen (registro) |

**Decisiones de diseño relevantes para el experimento**
- **Límites de recursos por pod** (`requests` 250m/256Mi, `limits` 500m/512Mi): hacen que cada réplica tenga una capacidad acotada, de modo que añadir réplicas aumente la capacidad total y el efecto sea medible.
- **`topologySpreadConstraints`** (`maxSkew: 1` por nodo): reparte las réplicas de forma uniforme entre los nodos disponibles, de modo que con 2 o 3 nodos los pods no se acumulan en uno solo.
- **PostgreSQL fijo** (preferencia por el nodo `role=database`): la base de datos es un único punto compartido por todas las réplicas, por lo que puede llegar a ser el cuello de botella.
- **Sondas:** `startupProbe` sobre `/ready` (espera a la BD); *liveness* y *readiness* sobre `/health`, que responde aunque el pod esté saturado.
- **Cómo se varían las réplicas:** `kubectl -n tienda scale deployment/api --replicas=N` (automatizado en `scripts/run_k8s_matrix.sh`).

In [ ]:
show_file("k8s/04-api.yaml", "yaml")
show_file("k8s/kustomization.yaml", "yaml")

### 4.3 Pasos de despliegue de cada escenario

```bash
# --- Fase 1: Docker Compose (máquina A) ---
docker compose up -d --build
curl http://<ip-maquina-A>:8000/reports/counts

# --- Fases 2-4: Kubernetes (K3s) ---
# 1) Imagen en un registro accesible por todos los nodos
./scripts/build_push.sh <usuario-dockerhub>          # y editar images.newName en k8s/kustomization.yaml
# 2) Un nodo:      instalar K3s server            -> docs/guia_k3s_multinodo.md
#    Dos nodos:    unir el nodo 2 (K3s agent)
#    Tres nodos:   unir el nodo 3 (K3s agent)
kubectl label node <nodo1> role=database
kubectl apply -k k8s/
kubectl -n tienda wait --for=condition=complete job/tienda-seed --timeout=300s
# 3) Medición por escenario (cambia las réplicas 1, 2, 3 y lanza JMeter en cada una)
./scripts/run_k8s_matrix.sh k8s-1nodo   <ip-nodo> 30080
```

## 5. Monitorización (opcional)

Además de las métricas de JMeter, la aplicación expone `/metrics` en formato Prometheus (peticiones por ruta/estado,
histograma de latencia, peticiones en curso) y cada respuesta incluye la cabecera `X-Served-By` con el nombre del pod/contenedor
que la atendió. La configuración está en `monitoring/` y `docker-compose.monitoring.yml` (Compose) o en el chart
`kube-prometheus-stack` (Kubernetes); ver `monitoring/README.md`.

> Si **no** usaste Prometheus/Grafana, indícalo aquí y salta la celda siguiente. Si lo usaste, pega capturas de Grafana y
> describe qué paneles observaste (CPU por pod, memoria, CPU por nodo, peticiones/s).

La siguiente celda es opcional: consulta la API de Prometheus y grafica la CPU por pod durante una prueba concreta.

In [ ]:
PROM_URL = None   # ej.: "http://<ip-nodo>:9090"  (con kubectl port-forward o NodePort). None = omitir.


def prom_query_range(base_url, query, start_s, end_s, step=5):
    qs = urllib.parse.urlencode({"query": query, "start": start_s, "end": end_s, "step": step})
    with urllib.request.urlopen(f"{base_url}/api/v1/query_range?{qs}", timeout=30) as resp:
        data = json.load(resp)
    rows = []
    for series in data["data"]["result"]:
        label = ",".join(f"{v}" for k, v in series["metric"].items() if k in ("pod", "instance", "name"))
        for ts, val in series["values"]:
            rows.append({"serie": label or "total", "tiempo": pd.to_datetime(float(ts), unit="s"), "valor": float(val)})
    return pd.DataFrame(rows)


def plot_prom_cpu(scenario, replicas, users, results_dir=None):
    "Grafica la CPU (núcleos) por pod durante la prueba (scenario, replicas, users)."
    path = Path(results_dir or RESULTS_DIR) / f"{scenario}__r{replicas}__u{users}.jtl"
    ts = pd.read_csv(path, usecols=["timeStamp"])["timeStamp"]
    start, end = ts.min() / 1000, ts.max() / 1000
    q = 'sum(rate(container_cpu_usage_seconds_total{namespace="tienda",container="api"}[30s])) by (pod)'
    df = prom_query_range(PROM_URL, q, start, end)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for name, g in df.groupby("serie"):
        ax.plot(g["tiempo"], g["valor"], label=name)
    ax.axhline(0.5, color="red", ls="--", lw=1, label="límite (0,5 CPU)")
    ax.set(title=f"CPU por pod - {scenario}, {replicas} réplica(s), {users} usuarios", ylabel="núcleos")
    ax.legend(fontsize=7)
    plt.show()


if PROM_URL:
    plot_prom_cpu("k8s-1nodo", 1, 100)      # ajusta a una prueba que hayas realizado
else:
    print("Prometheus no configurado: celda omitida.")

## 6. Metodología de carga con JMeter

**Herramienta:** Apache JMeter en modo *no-GUI*, ejecutado desde una máquina **distinta** de la que aloja la aplicación
(`scripts/run_jmeter.sh`). El plan `jmeter/plan_tienda.jmx` modela usuarios concurrentes en bucle cerrado
(cada usuario envía una petición, espera `think_ms` y envía la siguiente).

**Mezcla de peticiones** (porcentaje de ejecución de cada controlador de rendimiento):

| Petición | % | Qué ejercita |
|---|---:|---|
| `GET /products/{id}` | 40 | Lectura por clave primaria |
| `GET /products` (paginado) | 20 | Lectura + `COUNT(*)` |
| `GET /products?category` | 10 | Lectura con filtro e índice |
| `GET /products/search` | 15 | Escaneo secuencial `ILIKE` |
| `POST /orders` | 10 | Escritura transaccional |
| `GET /reports/top-products` | 5 | Agregación pesada |

**Diseño del experimento**

| Factor | Niveles |
|---|---|
| Escenario | Docker Compose · K8s 1 nodo · K8s 2 nodos · K8s 3 nodos |
| Réplicas de la API | 1, 2, 3 (Compose: 1) |
| Usuarios concurrentes | 10, 50, 100, 200 |
| Espera entre peticiones (*think time*) | 100 ms |
| Duración de cada prueba | 150 s en total: 30 s de *ramp-up* (se **descartan**) + 120 s de régimen estable |
| Pausa entre pruebas | 30 s |

**Controles para que la comparación sea justa**
- Mismos límites de CPU/memoria por réplica en Compose y Kubernetes; misma imagen y mismos datos.
- La máquina de JMeter nunca es la que aloja la aplicación, y se verifica que no se sature su CPU.
- Se comprueba que los pods estén `Ready` y repartidos entre nodos antes de cada prueba (`results/*__pods.txt`).
- Las pruebas se ejecutan siempre en el mismo orden y sin otras cargas en la red.
- *(Recomendado)* Repetir cada prueba 2–3 veces (`RUN_ID=1,2,3`); el notebook promedia las repeticiones.

**Métricas** (calculadas sobre la ventana estable, a partir de los `.jtl`)
- **Tiempo medio de respuesta** = media de `elapsed` (ms). Se añaden p50/p95/p99 porque la media oculta las colas.
- **Throughput** = peticiones completadas ÷ tiempo transcurrido (req/s), igual que el *Aggregate Report* de JMeter.
- **Tasa de error** = % de muestras con `success = false`.

In [ ]:
mezcla = pd.DataFrame({
    "Petición": ["GET /products/{id}", "GET /products", "GET /products?category", "GET /products/search", "POST /orders", "GET /reports/top-products"],
    "% de ejecución": [40, 20, 10, 15, 10, 5],
})
print(mezcla.to_string(index=False))
print("\nEjemplo de ejecución de una prueba:")
print("  ./scripts/run_jmeter.sh k8s-2nodos <ip-nodo> 30080 3 10 50 100 200")
print("\nComando JMeter equivalente para una sola prueba:")
print("  jmeter -n -t jmeter/plan_tienda.jmx -Jhost=<ip> -Jport=30080 -Jthreads=100 -Jrampup=30 "
      "-Jduration=150 -Jthink_ms=100 -l results/k8s-2nodos__r3__u100.jtl")

## 7. Carga de los resultados

Los archivos `results/<escenario>__r<réplicas>__u<usuarios>[__run<n>].jtl` son los CSV que genera JMeter. Para cada uno
se descartan los primeros `WARMUP_S` segundos (*ramp-up*) y se calculan las métricas de la ventana estable.

In [ ]:
FILE_RE = re.compile(r"^(?P<scenario>.+?)__r(?P<replicas>\d+)__u(?P<users>\d+)(?:__run(?P<run>\w+))?\.jtl$")
SCENARIO_ORDER = ["compose", "k8s-1nodo", "k8s-2nodos", "k8s-3nodos"]
SCENARIO_LABEL = {
    "compose": "Docker Compose",
    "k8s-1nodo": "K8s - 1 nodo",
    "k8s-2nodos": "K8s - 2 nodos",
    "k8s-3nodos": "K8s - 3 nodos",
}
CMAP = plt.get_cmap("tab10")
rcolor = lambda r: CMAP((int(r) - 1) % 10)


def read_jtl(path):
    df = pd.read_csv(path, usecols=["timeStamp", "elapsed", "label", "responseCode", "success"],
                     dtype={"responseCode": "string"})
    df["success"] = df["success"].astype(str).str.lower().eq("true")
    return df


def steady_window(df):
    t0 = df["timeStamp"].min()
    steady = df[df["timeStamp"] >= t0 + WARMUP_S * 1000]
    return steady if len(steady) else df


def compute_metrics(df):
    span_s = max((df["timeStamp"] + df["elapsed"]).max() - df["timeStamp"].min(), 1) / 1000.0
    el = df["elapsed"]
    return {
        "requests": len(df),
        "throughput_rps": len(df) / span_s,
        "avg_ms": el.mean(),
        "p50_ms": el.quantile(0.50),
        "p95_ms": el.quantile(0.95),
        "p99_ms": el.quantile(0.99),
        "error_pct": 100.0 * (1.0 - df["success"].mean()),
    }


run_rows, label_rows = [], []
files = sorted(Path(RESULTS_DIR).glob("*.jtl"))
if not files:
    raise FileNotFoundError(f"No hay archivos .jtl en '{RESULTS_DIR}/'. Ejecuta scripts/run_jmeter.sh o activa USE_DEMO_DATA.")

for path in files:
    m = FILE_RE.match(path.name)
    if not m:
        print("Se omite (nombre no reconocido):", path.name)
        continue
    key = {"scenario": m["scenario"], "replicas": int(m["replicas"]), "users": int(m["users"]), "run": m["run"] or "1"}
    steady = steady_window(read_jtl(path))
    run_rows.append({**key, **compute_metrics(steady)})
    for label, g in steady.groupby("label"):
        label_rows.append({**key, "label": label, **compute_metrics(g)})

runs = pd.DataFrame(run_rows)
by_label = pd.DataFrame(label_rows)
extra = [s for s in runs["scenario"].unique() if s not in SCENARIO_ORDER]
ORDER = [s for s in SCENARIO_ORDER if s in set(runs["scenario"])] + extra


def aggregate(df, keys):
    out = df.drop(columns=["run"]).groupby(keys, as_index=False).mean(numeric_only=True)
    out["scenario"] = pd.Categorical(out["scenario"], categories=ORDER, ordered=True)
    return out.sort_values(keys).reset_index(drop=True)


summary = aggregate(runs, ["scenario", "replicas", "users"])
summary["n_runs"] = runs.groupby(["scenario", "replicas", "users"]).size().values
by_label_s = aggregate(by_label, ["scenario", "replicas", "users", "label"])
MAX_USERS = int(summary["users"].max())
print(f"{len(files)} archivos .jtl | escenarios: {ORDER} | usuarios: {sorted(int(u) for u in summary['users'].unique())}")
summary

## 8. Resultados: tiempo medio de respuesta y throughput

### 8.1 Tiempo medio de respuesta frente a la carga

In [ ]:
def lines_by_scenario(metric, ylabel, title):
    fig, axes = plt.subplots(1, len(ORDER), figsize=(4.6 * len(ORDER), 4), sharey=True, squeeze=False)
    for ax, sc in zip(axes[0], ORDER):
        d = summary[summary["scenario"] == sc]
        for r, g in d.groupby("replicas"):
            g = g.sort_values("users")
            ax.plot(g["users"], g[metric], marker="o", color=rcolor(r), label=f"{r} réplica(s)")
        ax.set_title(SCENARIO_LABEL.get(sc, sc))
        ax.set_xlabel("Usuarios concurrentes")
        ax.set_xticks(sorted(summary["users"].unique()))
        ax.legend(fontsize=8)
    axes[0][0].set_ylabel(ylabel)
    fig.suptitle(title + TAG, y=1.02)
    fig.tight_layout()
    plt.show()


lines_by_scenario("avg_ms", "Tiempo medio de respuesta (ms)", "Tiempo medio de respuesta vs. usuarios concurrentes")

### 8.2 Throughput frente a la carga

In [ ]:
lines_by_scenario("throughput_rps", "Throughput (req/s)", "Throughput vs. usuarios concurrentes")

### 8.3 Latencia en la cola (p95) y errores
La media puede ocultar que una parte de los usuarios espera mucho más. El percentil 95 muestra esa cola.

In [ ]:
lines_by_scenario("p95_ms", "Percentil 95 (ms)", "Percentil 95 del tiempo de respuesta")

err = summary.pivot_table(index=["scenario", "replicas"], columns="users", values="error_pct", observed=True)
print("Tasa de error (%) por escenario, réplicas y usuarios:")
err

### 8.4 Un nodo, dos nodos y tres nodos: comparación directa
Mismas réplicas y mismo nivel de carga (el más alto probado), distinto número de nodos.

In [ ]:
def grouped_bars(metric, ylabel, users):
    d = summary[summary["users"] == users]
    reps = sorted(d["replicas"].unique())
    width = 0.8 / len(ORDER)
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    for i, sc in enumerate(ORDER):
        vals = [d[(d["scenario"] == sc) & (d["replicas"] == r)][metric].mean() for r in reps]
        bars = ax.bar(np.arange(len(reps)) + i * width, vals, width, label=SCENARIO_LABEL.get(sc, sc))
        for b, v in zip(bars, vals):
            if not np.isnan(v):
                ax.annotate(f"{v:.0f}", (b.get_x() + b.get_width() / 2, v), ha="center", va="bottom", fontsize=8)
    ax.set_xticks(np.arange(len(reps)) + width * (len(ORDER) - 1) / 2)
    ax.set_xticklabels([f"{r} réplica(s)" for r in reps])
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} con {users} usuarios concurrentes{TAG}")
    ax.legend()
    plt.show()


grouped_bars("throughput_rps", "Throughput (req/s)", MAX_USERS)
grouped_bars("avg_ms", "Tiempo medio de respuesta (ms)", MAX_USERS)

### 8.5 Escalabilidad horizontal: *speedup* y eficiencia
`speedup(r) = throughput(r réplicas) / throughput(1 réplica)` con la misma carga. El ideal es lineal (`speedup = r`);
`eficiencia = speedup / r`.

In [ ]:
def scalability_table(users):
    rows = []
    for sc in ORDER:
        d = summary[(summary["scenario"] == sc) & (summary["users"] == users)].set_index("replicas")
        if 1 not in d.index or len(d) < 2:
            continue
        base = d.loc[1, "throughput_rps"]
        for r, row in d.iterrows():
            rows.append({"escenario": SCENARIO_LABEL.get(sc, sc), "réplicas": r,
                         "throughput_rps": row["throughput_rps"], "avg_ms": row["avg_ms"],
                         "speedup": row["throughput_rps"] / base,
                         "eficiencia_%": 100 * row["throughput_rps"] / base / r})
    return pd.DataFrame(rows)


scal = scalability_table(MAX_USERS)
try:
    from IPython.display import display
except Exception:
    display = print
display(scal)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for esc, g in scal.groupby("escenario", sort=False):
    ax.plot(g["réplicas"], g["speedup"], marker="o", label=esc)
max_r = int(scal["réplicas"].max()) if len(scal) else 3
ax.plot(range(1, max_r + 1), range(1, max_r + 1), "k--", lw=1, label="ideal (lineal)")
ax.set(xlabel="Réplicas de la API", ylabel="Speedup del throughput", title=f"Escalabilidad horizontal con {MAX_USERS} usuarios{TAG}")
ax.set_xticks(range(1, max_r + 1))
ax.legend()
plt.show()

### 8.6 Mapas de calor del throughput (réplicas × usuarios)

In [ ]:
fig, axes = plt.subplots(1, len(ORDER), figsize=(4.4 * len(ORDER), 3.4), squeeze=False)
vmax = summary["throughput_rps"].max()
for ax, sc in zip(axes[0], ORDER):
    pv = summary[summary["scenario"] == sc].pivot_table(index="replicas", columns="users", values="throughput_rps", observed=True)
    sns.heatmap(pv, annot=True, fmt=".0f", cmap="YlGnBu", vmin=0, vmax=vmax, cbar=False, ax=ax)
    ax.set_title(SCENARIO_LABEL.get(sc, sc))
    ax.set_xlabel("Usuarios")
    ax.set_ylabel("Réplicas")
fig.suptitle("Throughput (req/s)" + TAG, y=1.03)
fig.tight_layout()
plt.show()

### 8.7 Punto de saturación
Se considera que una configuración se **satura** cuando duplicar o aumentar la carga al siguiente nivel mejora el
throughput menos de un 10 %.

In [ ]:
def saturation_table(tol=0.10):
    rows = []
    for (sc, r), g in summary.groupby(["scenario", "replicas"], observed=True):
        g = g.sort_values("users")
        x, u = g["throughput_rps"].values, g["users"].values
        sat = next((int(u[i - 1]) for i in range(1, len(x)) if x[i] < x[i - 1] * (1 + tol)), None)
        rows.append({"escenario": SCENARIO_LABEL.get(sc, sc), "réplicas": r,
                     "saturación (usuarios)": sat if sat is not None else f"> {int(u.max())}",
                     "throughput máx. (req/s)": x.max()})
    return pd.DataFrame(rows)


saturation_table()

### 8.8 ¿Qué endpoints sufren más?
Tiempo medio por tipo de petición en la carga máxima, con el mayor número de réplicas de cada escenario.

In [ ]:
top = by_label_s[by_label_s["users"] == MAX_USERS]
best = top.groupby("scenario", observed=True)["replicas"].max().rename("rmax").reset_index()
top = top.merge(best, on="scenario")
top = top[top["replicas"] == top["rmax"]]
pv = top.pivot_table(index="label", columns="scenario", values="avg_ms", observed=True)
pv = pv.reindex(columns=[s for s in ORDER if s in pv.columns])
pv.columns = [SCENARIO_LABEL.get(c, c) for c in pv.columns]
ax = pv.sort_values(pv.columns[0]).plot.barh(figsize=(8.5, 4.5))
ax.set(xlabel="Tiempo medio (ms)", ylabel="", title=f"Tiempo medio por endpoint con {MAX_USERS} usuarios (máx. réplicas){TAG}")
plt.tight_layout()
plt.show()

### 8.9 Hallazgos calculados automáticamente
Resumen numérico generado a partir de **tus** datos, como punto de partida para la sección de análisis.

In [ ]:
def hallazgos():
    out = []
    best = summary.loc[summary["throughput_rps"].idxmax()]
    out.append(f"Mayor throughput observado: {best['throughput_rps']:.0f} req/s en {SCENARIO_LABEL.get(best['scenario'], best['scenario'])} "
               f"con {int(best['replicas'])} réplica(s) y {int(best['users'])} usuarios (media {best['avg_ms']:.0f} ms).")
    for sc in ORDER:
        d = summary[(summary["scenario"] == sc) & (summary["users"] == MAX_USERS)].set_index("replicas")
        if len(d) >= 2 and 1 in d.index:
            rmax = d.index.max()
            sp = d.loc[rmax, "throughput_rps"] / d.loc[1, "throughput_rps"]
            dl = 100 * (d.loc[rmax, "avg_ms"] / d.loc[1, "avg_ms"] - 1)
            out.append(f"{SCENARIO_LABEL.get(sc, sc)}: pasar de 1 a {rmax} réplicas con {MAX_USERS} usuarios multiplica el throughput por "
                       f"{sp:.2f} (eficiencia {100 * sp / rmax:.0f} %) y el tiempo medio cambia {dl:+.0f} %.")
    for r in sorted(summary["replicas"].unique()):
        d = summary[(summary["replicas"] == r) & (summary["users"] == MAX_USERS) & (summary["scenario"].isin([s for s in ORDER if s.startswith('k8s')]))]
        if d["scenario"].nunique() >= 2:
            tt = d.set_index("scenario")["throughput_rps"].dropna()
            first, last = tt.index[0], tt.index[-1]
            out.append(f"Con {r} réplica(s) y {MAX_USERS} usuarios: {SCENARIO_LABEL.get(first, first)} = {tt.iloc[0]:.0f} req/s; "
                       f"{SCENARIO_LABEL.get(last, last)} = {tt.iloc[-1]:.0f} req/s ({100 * (tt.iloc[-1] / tt.iloc[0] - 1):+.0f} %).")
    if "compose" in ORDER and "k8s-1nodo" in ORDER:
        a = summary[(summary["scenario"] == "compose") & (summary["users"] == MAX_USERS) & (summary["replicas"] == 1)]
        b = summary[(summary["scenario"] == "k8s-1nodo") & (summary["users"] == MAX_USERS) & (summary["replicas"] == 1)]
        if len(a) and len(b):
            out.append(f"Compose vs. K8s (1 nodo, 1 réplica, {MAX_USERS} usuarios): {a['throughput_rps'].iloc[0]:.0f} vs. {b['throughput_rps'].iloc[0]:.0f} req/s.")
    worst_err = summary.loc[summary["error_pct"].idxmax()]
    out.append(f"Mayor tasa de error: {worst_err['error_pct']:.2f} % ({SCENARIO_LABEL.get(worst_err['scenario'], worst_err['scenario'])}, "
               f"{int(worst_err['replicas'])} réplica(s), {int(worst_err['users'])} usuarios).")
    return out


for line in hallazgos():
    print("-", line)

### 8.10 Recursos (opcional: Prometheus/Grafana o `docker stats` / `kubectl top`)

> Si recolectaste métricas internas, inserta aquí las capturas o gráficos (CPU y memoria por pod/nodo) de cada
> configuración y explica si la CPU de los pods llegó a su límite de 0,5 núcleos (indicador de saturación de la API)
> o si el cuello de botella estuvo en PostgreSQL.

## 9. Análisis comparativo

> **Esta sección debe escribirla el equipo con base en sus propios datos.** Las preguntas guía ayudan a estructurarla;
> cita siempre cifras concretas de las tablas y gráficos de la sección 8.

**9.1 Docker Compose vs. Kubernetes con un nodo y una réplica.** ¿Hay diferencia de rendimiento entre ambos? Si la hay,
¿a qué se debe (red del clúster, `kube-proxy`, límites de recursos, sobrecarga del orquestador)? `[COMPLETAR]`

**9.2 Efecto de las réplicas en un solo nodo.** ¿El throughput crece al pasar de 1 a 2 y a 3 réplicas? ¿Desde qué punto deja
de crecer y por qué (CPU del nodo compartida con PostgreSQL)? `[COMPLETAR]`

**9.3 Efecto del número de nodos.** Con las mismas réplicas, ¿cuánto mejoran throughput y tiempo de respuesta al pasar de
1 a 2 y a 3 nodos? ¿La mejora es proporcional? `[COMPLETAR]`

**9.4 Tendencias y puntos de saturación.** ¿A partir de cuántos usuarios se satura cada configuración? ¿Cómo evoluciona el
tiempo de respuesta y el p95 una vez saturada? `[COMPLETAR]`

**9.5 Cuellos de botella.** ¿La API o la base de datos limitó el rendimiento? Apóyate en la tabla por endpoint
(`/reports/top-products` y `/products/search` cargan más la BD), en las tasas de error y, si las tienes, en las métricas de CPU. `[COMPLETAR]`

**9.6 Eficiencia de recursos.** ¿Cuánto throughput adicional se obtiene por cada réplica o nodo adicional (eficiencia de la
sección 8.5)? ¿Compensa el costo de recursos? `[COMPLETAR]`

## 10. Conclusiones

> Redacta de 4 a 6 conclusiones **claras y verificables**, cada una respaldada por un dato. Contrasta explícitamente las
> hipótesis de la sección 1 (¿se confirmaron, se refutaron o se confirmaron parcialmente?).

1. `[COMPLETAR: Conclusión sobre Compose vs. Kubernetes]`
2. `[COMPLETAR: Conclusión sobre el escalado con réplicas en un nodo]`
3. `[COMPLETAR: Conclusión sobre 1 vs. 2 vs. 3 nodos]`
4. `[COMPLETAR: Conclusión sobre cuellos de botella y saturación]`
5. `[COMPLETAR: Conclusión sobre eficiencia de recursos]`

| Hipótesis | Resultado (confirmada / refutada / parcial) | Evidencia |
|---|---|---|
| 1. El throughput crece con las réplicas mientras haya CPU libre | `[COMPLETAR]` | `[COMPLETAR]` |
| 2. En un nodo, más réplicas que núcleos aportan poco | `[COMPLETAR]` | `[COMPLETAR]` |
| 3. Con más nodos escala más hasta que la BD limita | `[COMPLETAR]` | `[COMPLETAR]` |
| 4. Saturado el sistema, el tiempo crece ~linealmente con los usuarios | `[COMPLETAR]` | `[COMPLETAR]` |

## 11. Recomendaciones

> Deben derivarse de los resultados. Algunas líneas de trabajo que suelen surgir en este tipo de análisis (valida cuáles
> aplican a **tus** datos antes de incluirlas):

- Dimensionar réplicas y nodos según el punto de saturación observado, no según lo máximo posible.
- Si la BD fue el cuello de botella: añadir índices (p. ej. `pg_trgm` para `/products/search`), cachear los reportes o usar réplicas de lectura.
- Configurar un `HorizontalPodAutoscaler` basado en CPU, usando como umbral lo medido aquí.
- Ajustar `requests/limits` y el tamaño del pool de conexiones (`POOL_MAX_SIZE`) a la capacidad real del nodo.
- Repetir el experimento con más repeticiones y mayor duración para obtener intervalos de confianza.

`[COMPLETAR: recomendaciones propias basadas en tus resultados]`

## 12. Limitaciones y amenazas a la validez

- **Pocas repeticiones:** con una sola ejecución por configuración, la variabilidad natural puede confundirse con un efecto real.
- **Conexiones persistentes (*keep-alive*):** `kube-proxy` reparte por conexión, no por petición; con pocos usuarios las conexiones
  pueden quedar desigualmente distribuidas entre réplicas. Puedes comprobarlo con `-Jconnection=close` o consultando `/info`.
- **Datos que crecen:** cada `POST /orders` añade filas; tras muchas pruebas la BD es ligeramente mayor que al inicio.
- **Entorno compartido:** si las máquinas son virtuales o de un proveedor cloud, el rendimiento puede variar por vecinos ruidosos.
- **Carga sintética:** la mezcla de peticiones es un modelo; el comportamiento real de usuarios puede diferir.
- **PostgreSQL de una sola instancia:** el escalado de la capa de datos no forma parte de este estudio.